# Lesson 1.2 — Controller Architecture and Account Setup

## Learning Objectives
- Describe the architecture of the AppDynamics Controller Tenant
- Navigate to and log in to the Controller Tenant UI
- Understand the role of the Events Service and Analytics Service
- Make your first REST API call to the AppDynamics Controller

**Reference**: [Getting Started](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/getting-started) | [Controller Tenant UI](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/getting-started/controller-tenant-ui)

## 1. Controller Tenant Architecture

The **Controller Tenant** is the heart of AppDynamics. Think of it as the mission control center for your applications.

```
┌──────────────────────────────────────────────┐
│              Controller Tenant                │
│  ┌─────────────────┐  ┌────────────────────┐ │
│  │  Metrics Store   │  │   Events Service   │ │
│  │  (time-series)   │  │  (document store)  │ │
│  └─────────────────┘  └────────────────────┘ │
│  ┌─────────────────────────────────────────┐  │
│  │        Controller UI (web interface)    │  │
│  └─────────────────────────────────────────┘  │
│  ┌─────────────────────────────────────────┐  │
│  │         REST API Layer                  │  │
│  └─────────────────────────────────────────┘  │
└──────────────────────────────────────────────┘
         ↑ Agents send data here ↑
```

**Key components:**

| Component | Role |
|-----------|------|
| **Metrics Store** | Stores time-series performance metrics (response time, error rate, etc.) |
| **Events Service** | High-volume document store for Analytics data (transaction details, logs, BT data) |
| **Controller UI** | Web interface for dashboards, configuration, and exploration |
| **REST API** | Programmatic access to all metrics and configurations |
| **Account Management Portal** | Manage users, licenses, accounts at `accounts.appdynamics.com` |

## 2. Accessing the Controller Tenant UI

### Step-by-step: First Login

1. Go to the [Account Management Portal](https://accounts.appdynamics.com)
2. Sign in with your Splunk/AppDynamics credentials
3. Navigate to your Controller Tenant URL:
   - SaaS format: `https://<your-account-name>.saas.appdynamics.com`
   - Example: `https://acme-corp.saas.appdynamics.com`
4. Log in with your **account name**, **username**, and **password**

### UI Sections to Know

| Section | What You'll Find |
|---------|------------------|
| **Home** | Overview of all monitored applications |
| **Applications** | Your APM applications — tier/node topology, BT list |
| **Dashboards & Reports** | Custom dashboards and Dash Studio |
| **Analytics** | ADQL query interface (Business iQ) |
| **Alert & Respond** | Health rules, policies, actions |
| **Settings** | Agent management, users, license |
| **Metric Browser** | Browse the raw metric tree |

In [ ]:
# Setup: Store your credentials securely as environment variables
# Run this cell ONCE to set up — replace placeholder values
# In production, use a secrets manager or .env file

import os

# ── Set these before running ──────────────────────────────────────────────────
CONTROLLER_URL = os.getenv("APPD_CONTROLLER_URL", "https://your-account.saas.appdynamics.com")
ACCOUNT_NAME   = os.getenv("APPD_ACCOUNT",        "your-account-name")
USERNAME       = os.getenv("APPD_USER",            "your-username")
PASSWORD       = os.getenv("APPD_PASSWORD",        "your-password")
# ─────────────────────────────────────────────────────────────────────────────

print("Configuration loaded:")
print(f"  Controller URL : {CONTROLLER_URL}")
print(f"  Account Name   : {ACCOUNT_NAME}")
print(f"  Username       : {USERNAME}")
print(f"  Password       : {'*' * len(PASSWORD) if PASSWORD else '(not set)'}")

## 3. The AppDynamics REST API

AppDynamics exposes a comprehensive REST API. You can use it to:
- Retrieve metrics programmatically
- List applications, tiers, business transactions
- Export configuration
- Trigger actions and deployments

**Authentication**: The API uses HTTP Basic Auth with the format:
```
username@account-name : password
```

For example: `admin@acme-corp:mypassword`

**Base URL pattern**:
```
GET https://<controller>/controller/rest/<resource>?output=JSON
```

**Reference**: [AppDynamics REST APIs](https://help.splunk.com/en/appdynamics-saas/extend-splunk-appdynamics/26.2.0/extend-splunk-appdynamics/splunk-appdynamics-apis)

In [ ]:
import requests
import json

def appd_get(endpoint: str) -> dict | list:
    """
    Helper: make an authenticated GET request to the AppDynamics REST API.
    
    endpoint: path after /controller/rest/ (e.g. 'applications')
    Returns: parsed JSON response
    """
    url = f"{CONTROLLER_URL}/controller/rest/{endpoint}"
    auth = (f"{USERNAME}@{ACCOUNT_NAME}", PASSWORD)
    params = {"output": "JSON"}
    
    response = requests.get(url, auth=auth, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

print("Helper function 'appd_get' defined.")
print("Usage: appd_get('applications') — lists all monitored applications")

In [ ]:
# API Call 1: List all applications
# This is the simplest API call — it confirms your credentials work.

# NOTE: This will fail if CONTROLLER_URL / credentials are not real.
# Replace the env vars at the top of this notebook with your actual values first.

try:
    applications = appd_get("applications")
    print(f"Found {len(applications)} application(s):")
    for app in applications:
        print(f"  ID: {app['id']:5d}  Name: {app['name']}")
except requests.exceptions.ConnectionError:
    print("[Connection Error] Could not reach the Controller.")
    print("  → Make sure APPD_CONTROLLER_URL is set correctly")
    print("  → Sample output shown below:")
    # Simulated output for learning purposes
    sample = [{"id": 1, "name": "StoreFront"}, {"id": 2, "name": "PaymentService"}]
    for app in sample:
        print(f"  ID: {app['id']:5d}  Name: {app['name']}")
except requests.exceptions.HTTPError as e:
    print(f"[HTTP Error] {e}")
    print("  → Check your username, password, and account name")

## 4. Understanding Multi-Tenancy

AppDynamics SaaS is **multi-tenant**: many customers share the same infrastructure but each has an isolated **Controller Tenant**.

Key isolation points:
- Each tenant has its own **account name** — used in API auth (`user@account`)
- Data is partitioned per account — you cannot see another tenant's data
- One account can host **multiple applications** (e.g., StoreFront, PaymentService, Mobile)

The **Account Management Portal** (`accounts.appdynamics.com`) lets you:
- Manage users and roles
- View license consumption
- Download agents
- Configure SSO

In [ ]:
# EXERCISE 1.2a: Verify API connectivity
# 
# 1. Set your environment variables (APPD_CONTROLLER_URL, APPD_ACCOUNT, APPD_USER, APPD_PASSWORD)
# 2. Run the cell below to list your applications
# 3. Note the application IDs — you'll use them in later lessons
#
# TASK: Fill in your_app_id below with the ID of your first application

your_app_id = None  # TODO: replace None with your app ID (integer)

if your_app_id:
    try:
        tiers = appd_get(f"applications/{your_app_id}/tiers")
        print(f"Tiers in application {your_app_id}:")
        for tier in tiers:
            print(f"  Tier: {tier['name']} (ID: {tier['id']}, Agent: {tier.get('agentType', 'N/A')})")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Set your_app_id to list tiers.")
    print("Example output:")
    print("  Tier: WebTier (ID: 101, Agent: JAVA)")
    print("  Tier: PaymentService (ID: 102, Agent: JAVA)")
    print("  Tier: OrderDB-Backend (ID: 103, Agent: NONE)")

## 5. The Events Service — Your Analytics Back-end

The **Events Service** is a separate document-oriented store optimized for high-volume, schema-flexible data. It stores:

- **Transaction analytics data** — detailed attributes of every Business Transaction
- **Log data** — application logs correlated to transactions
- **Browser RUM data** — page load records, AJAX call data
- **Mobile data** — crash records, network requests

You query the Events Service using **ADQL** (AppDynamics Query Language) — covered in Phase 3.

**Important distinction**:
- **Metrics Store** → queried by the Metric Browser and REST API → aggregate time-series
- **Events Service** → queried by ADQL → raw event records with full attribute detail

Think of it this way:
- Metrics Store: "Average response time was 320ms in the last 5 minutes"
- Events Service: "Here are the 1,240 individual transactions from the last 5 minutes, each with user ID, device type, and exact timing"

## Summary

- The **Controller Tenant** is the central hub: it receives agent data, runs the UI, and serves the REST API
- **Metrics Store** = aggregate time-series data; **Events Service** = detailed event records
- Authentication uses `username@account-name : password` for REST API calls
- The `appd_get()` helper you wrote will be used in later lessons to retrieve metrics programmatically

---

**Next**: [Lesson 1.3 — Applications, Tiers, and Nodes](lesson-1.3-applications-tiers-nodes.ipynb)